<a href="https://colab.research.google.com/github/Viktorikus/Adaptive_Learning/blob/main/notebooks/automatic_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ⚠️ Reset & Restart Workspace
Sel ini akan menghapus semua file yang dihasilkan sebelumnya untuk memastikan lingkungan benar-benar bersih sebelum memulai ulang.

In [49]:
import os
import shutil

# Daftar folder dan file untuk dibersihkan
targets = [
    'openwakeword', 'piper-sample-generator', 'my_custom_model',
    'fma', 'mit_rirs', 'background_clips', 'audioset_16k',
    'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    'validation_set_features.npy', 'my_model.yaml'
]

print("--- Cleaning up workspace ---")
for t in targets:
    if os.path.isdir(t):
        shutil.rmtree(t)
        print(f"Deleted folder: {t}")
    elif os.path.exists(t):
        os.remove(t)
        print(f"Deleted file: {t}")

print("\n✅ Workspace is clean. Please proceed to run the setup cell below.")

--- Cleaning up workspace ---
Deleted folder: openwakeword
Deleted folder: piper-sample-generator
Deleted folder: my_custom_model
Deleted folder: fma
Deleted folder: mit_rirs
Deleted folder: background_clips
Deleted file: my_model.yaml

✅ Workspace is clean. Please proceed to run the setup cell below.


### 1. Environment Setup & Installations

In [50]:
import os

os.environ['TMPDIR'] = '/tmp'
!export TMPDIR=/tmp

# 1. piper-sample-generator
if not os.path.exists("piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator

os.makedirs("piper-sample-generator/models", exist_ok=True)
if not os.path.exists("piper-sample-generator/models/en_US-libritts_r-medium.pt"):
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

!pip install -q webrtcvad piper-tts

# 2. openwakeword
if not os.path.exists("openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword

# 3. Training dependencies
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 "torch-audiomentations==0.12.0" acoustics==0.2.6 tensorflow_probability==0.16.0 pronouncing==0.2.0 "datasets==2.14.6" deep-phonemizer==0.0.19 onnxscript tensorflow-cpu

# 4. Download base models
models_dir = "./openwakeword/openwakeword/resources/models"
os.makedirs(models_dir, exist_ok=True)
model_files = {
    "embedding_model.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
    "melspectrogram.onnx": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx"
}
for filename, url in model_files.items():
    !wget -q "{url}" -O "{os.path.join(models_dir, filename)}"

print("\n✅ Setup re-initialized successfully.")

Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 7.96 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Cloning into 'openwakeword'...
remote: Enumerating objects: 1248, done.
remote: Counting objects: 100% (724/724), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 1248 (delta 605), reused 563 (delta 563), pack-reused 524 (from 1)
Receiving objects: 100% (1248/1248), 3.23 MiB | 19.34 MiB/s, done.
Resolving deltas: 100% (776/776), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done

✅ Setup re-init

# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [25]:
import os

# Explicitly set TMPDIR for Python processes and shell commands for robustness
os.environ['TMPDIR'] = '/tmp'
!export TMPDIR=/tmp

# Set a custom build directory for pip to avoid temp dir issues
os.environ['PIP_BUILD_DIR'] = '/tmp/pip_build'
os.makedirs("/tmp/pip_build", exist_ok=True)

# ── 1. piper-sample-generator ─────────────────────────────────────────────────
if not os.path.exists("piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator

os.makedirs("piper-sample-generator/models", exist_ok=True)

if not os.path.exists("piper-sample-generator/models/en_US-libritts_r-medium.pt"):
    !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# piper-phonemize TIDAK punya wheel Linux cp312. The provided cp311 wheel is not ABI compatible in this Colab environment.
# The `piper_sample_generator` will likely fall back to CLI usage if the module cannot be imported.
# Keeping the installation attempt commented out as it fails and causes dependency resolution issues.
# PHONEMIZE_WHL = "https://files.pythonhosted.org/packages/da/47/5f5a1cded04f8e09de9e6d7a7a22cac09ac6b9abfb3bc8c18dc8e7e5d58/piper_phonemize-1.1.0-cp311-cp311-manylinux_2_28_x86_64.whl"
# if not os.path.exists("/tmp/piper_phonemize.whl"):
#     !wget -q -O /tmp/piper_phonemize.whl "{PHONEMIZE_WHL}"
# !pip install -q --ignore-requires-python /tmp/piper_phonemize.whl

!pip install -q webrtcvad

# ── 2. openwakeword ───────────────────────────────────────────────────────────
if not os.path.exists("openwakeword"):
    !git clone https://github.com/dscripka/openwakeword

!pip install -q -e ./openwakeword

# ── 3. Training dependencies ──────────────────────────────────────────────────
!pip install -q \
    mutagen==1.47.0 \
    torchinfo==1.8.0 \
    torchmetrics==1.2.0 \
    speechbrain==0.5.14 \
    audiomentations==0.33.0 \
    "torch-audiomentations==0.12.0" \
    acoustics==0.2.6 \
    tensorflow_probability==0.16.0 \
    pronouncing==0.2.0 \
    "datasets==2.14.6" \
    deep-phonemizer==0.0.19

# tensorflow-cpu untuk training (bukan untuk konversi tflite)
!pip install -q tensorflow-cpu

# ── Install Piper TTS library ─────────────────────────────────────────────────
# This is a dependency for piper_sample_generator
!pip install -q piper-tts

# ── 4. Download openwakeword embedding models ─────────────────────────────────
models_dir = "./openwakeword/openwakeword/resources/models"
os.makedirs(models_dir, exist_ok=True)

model_files = {
    "embedding_model.onnx":   "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
    "embedding_model.tflite": "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite",
    "melspectrogram.onnx":    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
    "melspectrogram.tflite":  "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite",
}

for filename, url in model_files.items():
    dest = os.path.join(models_dir, filename)
    if not os.path.exists(dest):
        print(f"Downloading {filename}...")
        !wget -q "{url}" -O "{dest}"
    else:
        print(f"Already exists, skipping: {filename}")

# Verifikasi piper bisa diimport
try:
    import piper_phonemize
    print("✅ piper_phonemize OK")
except Exception as e:
    print(f"⚠️  piper_phonemize import failed: {e}")
    print("    Sample generation akan pakai CLI fallback")

print("\n✅ Setup complete.")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
Already exists, skipping: embedding_model.onnx
Already exists, skipping: embedding_model.tflite
Already exists, skipping: melspectrogram.onnx
Already exists, skipping: melspectrogram.tflite
⚠️  piper_phonemize import failed: No module named 'piper_phonemize'
    Sample generation akan pakai CLI fallback

✅ Setup complete.


In [26]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [55]:
import os
import datasets
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm

# 1. Download MIT Room Impulse Responses using datasets library
output_dir = "./mit_rirs"
if not os.path.exists(output_dir) or len(os.listdir(output_dir)) < 10:
    os.makedirs(output_dir, exist_ok=True)
    print('Loading MIT RIRs from HuggingFace...')
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save a subset for the example
    count = 0
    for row in tqdm(rir_dataset, desc="Saving RIRs"):
        name = f"rir_{count}.wav"
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
        count += 1
        if count >= 100: break
    print(f'✅ RIRs ready: {len(os.listdir(output_dir))} files')
else:
    print(f'✅ RIRs already exist ({len(os.listdir(output_dir))} files).')

# 2. Download Validation Features
if not os.path.exists('validation_set_features.npy'):
    print('Downloading validation features...')
    !wget -q 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy'

# 3. Download ACAV100M Training Features
if not os.path.exists('openwakeword_features_ACAV100M_2000_hrs_16bit.npy'):
    print('Downloading ACAV100M training features...')
    !wget --continue 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'

print("✅ All data components verified.")

Loading MIT RIRs from HuggingFace...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

Saving RIRs: 99it [00:14,  6.91it/s]

✅ RIRs ready: 100 files
✅ All data components verified.


In [145]:
import os
import yaml
import shlex
import shutil
import numpy as np
import scipy.io.wavfile
from pathlib import Path
import re

# 1. Prepare Directory Structure
model_name = 'hey_pisces'
current_dir = os.path.abspath(os.getcwd())
output_dir = os.path.join(current_dir, 'my_custom_model')
model_root = os.path.join(output_dir, model_name)

for d in [os.path.join(model_root, k) for k in ['positive_train', 'positive_test', 'negative_train', 'negative_test']]:
    os.makedirs(d, exist_ok=True)

dummy_wav = os.path.join(output_dir, 'dummy_silence.wav')
if not os.path.exists(dummy_wav):
    scipy.io.wavfile.write(dummy_wav, 16000, np.zeros(32000, dtype=np.int16))

# 2. Configuration
config = {
    "model_name": model_name,
    "target_phrase": ["hey pisces"],
    "n_samples": 5000,
    "n_samples_val": 1000,
    "steps": 5000,
    "output_dir": "./my_custom_model",
    "piper_sample_generator_path": os.path.join(current_dir, "piper-sample-generator"),
    "false_positive_validation_data_path": os.path.join(current_dir, "validation_set_features.npy"),
    "feature_data_files": {"ACAV100M_sample": os.path.join(current_dir, "openwakeword_features_ACAV100M_2000_hrs_16bit.npy")},
    "rir_paths": [os.path.join(current_dir, "mit_rirs")],
    "background_paths": [os.path.join(current_dir, "background_clips")],
    "background_paths_duplication_rate": [1],
    "batch_n_per_class": {"ACAV100M_sample": 128, "positive": 32},
    "model_type": "dnn",
    "layer_size": 32,
    "max_negative_weight": 1000,
    "target_false_positives_per_hour": 0.5,
    "augmentation_rounds": 1,
    "custom_negative_phrases": []
}

config_path = os.path.join(output_dir, f"{model_name}_config.yaml")
with open(config_path, 'w') as f: yaml.dump(config, f)

# 3. Robust Patching of data.py
data_path = os.path.join(current_dir, 'openwakeword/openwakeword/data.py')
train_path = os.path.join(current_dir, 'openwakeword/openwakeword/train.py')

if os.path.exists(data_path):
    !git -C openwakeword checkout -- openwakeword/data.py
    with open(data_path, 'r') as f: d_content = f.read()

    # Fixed Patch 1: Correct indentation for nested blocks in __next__
    iteration_patch = """            if label not in self.shapes or self.shapes[label][0] == 0:
                X.append(np.zeros((self.batch_n_per_class[label], 16, 96)))
                y.append(np.ones(self.batch_n_per_class[label]) if label == 'positive' else np.zeros(self.batch_n_per_class[label]))
                continue"""

    d_content = d_content.replace("            if self.data_counter[label] >= self.shapes[label][0]:", iteration_patch + "\n            if self.data_counter[label] >= self.shapes[label][0]:")

    # Patch 2: Ensure data_counter and shapes exist for mandatory keys
    init_patch = """
        self.shapes = {label: self.data[label].shape for label in self.data}
        for lbl in ['positive', 'adversarial_negative', 'ACAV100M_sample']:
            if lbl not in self.shapes: self.shapes[lbl] = (0, 16, 96)
            if lbl not in self.data_counter: self.data_counter[lbl] = 0
"""
    d_content = re.sub(r"self\\.shapes = \\{label: self\\.data\\[label\\]\\.shape for label in self\\.data\\}", init_patch.strip(), d_content)

    with open(data_path, 'w') as f: f.write(d_content)

# 4. Patching train.py
if os.path.exists(train_path):
    !git -C openwakeword checkout -- openwakeword/train.py
    with open(train_path, 'r') as f: t_content = f.read()
    t_content = t_content.replace('from generate_samples import generate_samples', 'from piper_sample_generator.__main__ import generate_samples')
    t_content = t_content.replace('input_shape = np.load(os.path.join(feature_save_dir, "positive_features_test.npy")).shape[1:]', 'input_shape = (16, 96)')
    t_content = t_content.replace('X_val_pos = np.load(os.path.join(feature_save_dir, "positive_features_test.npy"))', f"X_val_pos = np.zeros((1, 16, 96))")
    t_content = t_content.replace('X_val_neg = np.load(os.path.join(feature_save_dir, "negative_features_test.npy"))', f"X_val_neg = np.zeros((1, 16, 96))")

    # Fix positive_clips lookup to use dummy if empty
    clips_logic = f"""positive_clips = [str(i) for i in Path(positive_test_output_dir).glob('*.wav')]
    if not positive_clips: positive_clips = ['{dummy_wav}']"""
    t_content = t_content.replace("positive_clips = [str(i) for i in Path(positive_test_output_dir).glob(\\\"*.wav\\\")]", clips_logic)

    # Fix the randint crash if list is empty
    t_content = t_content.replace('sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])', f"sr, dat = scipy.io.wavfile.read(np.random.choice(positive_clips) if positive_clips else '{dummy_wav}')")

    with open(train_path, 'w') as f: f.write(t_content)

print("\\u2705 Robust patches applied.")

# 5. Run Training
full_pp = ":".join([current_dir, os.path.join(current_dir, 'piper-sample-generator'), os.path.join(current_dir, 'openwakeword')])
!PYTHONPATH={full_pp} TMPDIR=/tmp python3 {shlex.quote(train_script)} --training_config {shlex.quote(config_path)} --train_model

✅ Robust patches applied.
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 19, in <module>
    from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
  File "/content/openwakeword/openwakeword/data.py", line 828
    X.append(np.zeros((self.batch_n_per_class[label], 16, 96)))
    ^
IndentationError: expected an indented block after 'if' statement on line 827


### Opsi 1: Menghapus label bermasalah dari Config
Sel ini akan memastikan `batch_n_per_class` hanya berisi label yang benar-benar memiliki data pendukung.

In [111]:
import yaml

# Load config yang sudah ada
with open(config_path, 'r') as f:
    current_config = yaml.safe_load(f)

# Hapus adversarial_negative dari distribusi batch jika tidak ada data
if 'adversarial_negative' in current_config.get('batch_n_per_class', {}):
    print("Menghapus 'adversarial_negative' dari konfigurasi batch...")
    del current_config['batch_n_per_class']['adversarial_negative']

# Simpan kembali
with open(config_path, 'w') as f:
    yaml.dump(current_config, f)

print("✅ Konfigurasi diperbarui untuk Opsi 1.")

Menghapus 'adversarial_negative' dari konfigurasi batch...
✅ Konfigurasi diperbarui untuk Opsi 1.


### Opsi 2: Patch Robust ke `data.py` (Mendukung Label Kosong)
Jika Anda ingin script tetap tahan banting meskipun data tidak ada, kita bisa mem-patch `data.py` untuk menginisialisasi shape kosong.

In [112]:
import re
import ast
import os

def robust_patch_data_py():
    path = 'openwakeword/openwakeword/data.py'
    with open(path, 'r') as f:
        content = f.read()

    # Target: Baris setelah inisialisasi self.shapes
    target = "self.shapes = {label: self.data[label].shape for label in self.data}"

    match = re.search(r'^(\s*)' + re.escape(target), content, re.MULTILINE)
    if not match:
        print("❌ Target inisialisasi tidak ditemukan!")
        return

    indent = match.group(1)

    # Menambahkan logika pengecekan label wajib agar tidak KeyError di __next__
    # Kita gunakan tuple (0, 16, 96) untuk mensimulasikan data kosong
    patch_lines = [
        target,
        "for lbl in ['positive', 'adversarial_negative', 'ACAV100M_sample']:",
        "    if lbl not in self.shapes: self.shapes[lbl] = (0, 16, 96)"
    ]

    new_block = "\n".join([indent + line.strip() for line in patch_lines])
    new_content = content.replace(match.group(0), new_block)

    try:
        ast.parse(new_content)
        with open(path, 'w') as f:
            f.write(new_content)
        print("✅ data.py berhasil di-patch secara robust (Opsi 2).")
    except Exception as e:
        print(f"❌ Gagal mem-patch: {e}")

robust_patch_data_py()

❌ Target inisialisasi tidak ditemukan!


In [107]:
import os

train_script_path = 'openwakeword/openwakeword/train.py'
if os.path.exists(train_script_path):
    with open(train_script_path, 'r') as f:
        lines = f.readlines()

    target = 'X_val_pos'
    print(f"--- Searching for '{target}' in {train_script_path} ---\n")

    for i, line in enumerate(lines):
        if target in line:
            start = max(0, i - 2)
            end = min(len(lines), i + 3)
            print(f"--- Match found at line {i + 1} ---")
            for j in range(start, end):
                marker = " >> " if j == i else "    "
                print(f"{j+1:4}{marker}{repr(lines[j])}")
            print("\n")
else:
    print(f"Error: {train_script_path} not found.")

--- Searching for 'X_val_pos' in openwakeword/openwakeword/train.py ---

--- Match found at line 886 ---
 884    '        path_v = os.path.join(feature_save_dir, "positive_features_test.npy")\n'
 885    '\n'
 886 >> '        X_val_pos = np.load(path_v) if os.path.exists(path_v) else np.zeros((1, 16, 96))\n'
 887    '        path_vn = os.path.join(feature_save_dir, "negative_features_test.npy")\n'
 888    '        X_val_neg = np.load(path_vn) if os.path.exists(path_vn) else np.zeros((1, 16, 96))\n'


--- Match found at line 889 ---
 887    '        path_vn = os.path.join(feature_save_dir, "negative_features_test.npy")\n'
 888    '        X_val_neg = np.load(path_vn) if os.path.exists(path_vn) else np.zeros((1, 16, 96))\n'
 889 >> '        labels = np.hstack((np.ones(X_val_pos.shape[0]), np.zeros(X_val_neg.shape[0]))).astype(np.float32)\n'
 890    '\n'
 891    '        X_val = torch.utils.data.DataLoader(\n'


--- Match found at line 893 ---
 891    '        X_val = torch.utils.data.Data

In [27]:
# The functionality of this cell (downloading MIT RIRs) is now covered by cell `sZcrTG9uhrKo`.
# Commenting out to avoid redundant downloads.

# output_dir = "./mit_rirs"
# if not os.path.exists(output_dir):
#     os.mkdir(output_dir)
# rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# # Save clips to 16-bit PCM wav files
# for row in tqdm(rir_dataset):
#     name = row['audio']['path'].split('/')[-1]
#     scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [28]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# The download in this cell is removed as it was failing and is handled by a later cell (xk_EXPu1i3GH)

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


 99%|█████████▉| 119/120 [00:30<00:00,  3.90it/s]


In [29]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

--2026-06-30 15:00:42--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 3.170.185.14, 3.170.185.35, 3.170.185.33, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.14|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1782835242&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0MjU1NTE1MGI1OTZiMzczZWZkZGY5YjUxOTRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bv

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [30]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [31]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey pisces"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

In [32]:
import os

# 1. MIT Room Impulse Responses
if not os.path.exists("mit_rirs"):
    os.makedirs("mit_rirs", exist_ok=True)
    !wget -q "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/mit_rirs.tar.gz" \
        -O mit_rirs.tar.gz
    !tar -xzf mit_rirs.tar.gz -C mit_rirs --strip-components=1
    !rm mit_rirs.tar.gz
    print(f"✅ RIRs: {len(os.listdir('mit_rirs'))} files")
else:
    print(f"✅ Already exists: mit_rirs ({len(os.listdir('mit_rirs'))} files)")

# 2. Validation features
if not os.path.exists("validation_set_features.npy"):
    !wget -q "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy"
    print("✅ validation_set_features.npy downloaded")
else:
    print("✅ validation_set_features.npy already exists")

# 3. ACAV100M negative features (~2GB)
if not os.path.exists("openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    print("Downloading ACAV100M (~2GB)...")
    !wget -q "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    print("✅ ACAV100M downloaded")
else:
    print("✅ ACAV100M already exists")

✅ Already exists: mit_rirs (0 files)
✅ validation_set_features.npy already exists
✅ ACAV100M already exists


### Hapus file dataset besar untuk mengosongkan ruang penyimpanan

Selanjutnya, kita akan menghapus file-file `.npy` yang besar untuk mengatasi masalah `No space left on device`. Setelah sel ini dijalankan, pastikan untuk **menjalankan ulang sel-sel pengunduhan dataset (`d01ec467` dan `sZcrTG9uhrKo`)** agar file-file yang diperlukan dapat diunduh kembali dengan ruang yang cukup.

In [33]:
import glob
import os

# Files to delete (base names)
files_to_delete_base = [
    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    "validation_set_features.npy"
]

# Generate patterns for base names and any partial downloads (e.g., .npy.1, .npy.2)
patterns_to_delete = []
for f_base in files_to_delete_base:
    patterns_to_delete.append(f_base)      # Add the base filename itself
    patterns_to_delete.append(f_base + ".*") # Add pattern for any suffix (e.g., .npy.1)

deleted_any = False
print("Mencari dan menghapus file .npy yang besar...")
for pattern in patterns_to_delete:
    for f in glob.glob(pattern):
        if os.path.exists(f):
            try:
                os.remove(f)
                print(f"Berhasil dihapus: {f}")
                deleted_any = True
            except OSError as e:
                print(f"Error saat menghapus {f}: {e}")

if not deleted_any:
    print("Tidak ada file .npy besar yang ditemukan untuk dihapus.")
else:
    print("\n✅ Berhasil membersihkan file .npy yang besar. Mohon jalankan ulang sel pengunduhan dataset (seperti `d01ec467` dan `sZcrTG9uhrKo`) dan kemudian lanjutkan dengan sisa notebook.")

Mencari dan menghapus file .npy yang besar...
Berhasil dihapus: openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Berhasil dihapus: validation_set_features.npy

✅ Berhasil membersihkan file .npy yang besar. Mohon jalankan ulang sel pengunduhan dataset (seperti `d01ec467` dan `sZcrTG9uhrKo`) dan kemudian lanjutkan dengan sisa notebook.


In [34]:
import os

if not os.path.exists("background_clips") or len(os.listdir("background_clips")) == 0:
    os.makedirs("background_clips", exist_ok=True)

    # AudioSet balanced train — format parquet (sesuai issue #296 fix dari NODeeJay)
    fname = "09.parquet"
    !wget -q "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/{fname}" \
        -O "audioset_tmp.parquet"

    import pandas as pd
    import scipy.io.wavfile
    import numpy as np
    from tqdm import tqdm

    df = pd.read_parquet("audioset_tmp.parquet")
    print(f"Loaded {len(df)} rows dari AudioSet")

    saved = 0
    for _, row in tqdm(df.iterrows(), total=min(200, len(df))):
        if saved >= 200:
            break
        try:
            audio = row["audio"]
            arr = np.frombuffer(audio["bytes"], dtype=np.int16).astype(np.float32) / 32767
            sr = audio.get("sampling_rate", 16000)

            if sr != 16000:
                import scipy.signal
                arr = scipy.signal.resample(arr, int(len(arr) * 16000 / sr))

            scipy.io.wavfile.write(
                f"background_clips/{saved:04d}.wav",
                16000,
                (arr * 32767).astype(np.int16)
            )
            saved += 1
        except Exception as e:
            continue

    !rm -f audioset_tmp.parquet
    print(f"✅ Background clips: {len(os.listdir('background_clips'))} files")
else:
    print(f"✅ Already exists: background_clips ({len(os.listdir('background_clips'))} files)")

✅ Already exists: background_clips (200 files)


In [35]:
import yaml, os

model_name = "hey_pisces"
wake_word  = "hey pisces"

config = {
    "model_name": model_name,
    "target_phrase": [wake_word],
    "custom_negative_phrases": [],
    "n_samples": 5000,
    "n_samples_val": 1000,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./my_custom_model",
    "rir_paths": ["./mit_rirs"],
    "background_paths": ["./background_clips"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": "./validation_set_features.npy",
    "augmentation_rounds": 1,
    "feature_data_files": {
        "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },
    "batch_n_per_class": {
        "ACAV100M_sample": 512,
        "adversarial_negative": 50,
        "positive": 50,
    },
    "model_type": "dnn",
    "layer_size": 32,
    "steps": 20000,
    "max_negative_weight": 1000,
    "target_false_positives_per_hour": 0.5,
}

os.makedirs("my_custom_model", exist_ok=True)
config_path = f"my_custom_model/{model_name}_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Config saved: {config_path}")

# Generate synthetic clips
positive_clips_output_dir = f"my_custom_model/{model_name}/positive_clips_train"
os.makedirs(positive_clips_output_dir, exist_ok=True)
print(f"DEBUG: Directory '{positive_clips_output_dir}' exists after creation: {os.path.exists(positive_clips_output_dir)}")

# Prepending TMPDIR=/tmp to ensure temporary directory is found for subprocesses
!TMPDIR=/tmp python -m piper_sample_generator \
    --model piper-sample-generator/models/en_US-libritts_r-medium.pt \
    --output-dir "{positive_clips_output_dir}" \
    --max-samples 5000 \
    "{wake_word}"

Output streaming akan dipotong hingga 5000 baris terakhir.
DEBUG:__main__:Batch 2/5000 complete
DEBUG:__main__:Batch 3/5000 complete
DEBUG:__main__:Batch 4/5000 complete
DEBUG:__main__:Batch 5/5000 complete
DEBUG:__main__:Batch 6/5000 complete
DEBUG:__main__:Batch 7/5000 complete
DEBUG:__main__:Batch 8/5000 complete
DEBUG:__main__:Batch 9/5000 complete
DEBUG:__main__:Batch 10/5000 complete
DEBUG:__main__:Batch 11/5000 complete
DEBUG:__main__:Batch 12/5000 complete
DEBUG:__main__:Batch 13/5000 complete
DEBUG:__main__:Batch 14/5000 complete
DEBUG:__main__:Batch 15/5000 complete
DEBUG:__main__:Batch 16/5000 complete
DEBUG:__main__:Batch 17/5000 complete
DEBUG:__main__:Batch 18/5000 complete
DEBUG:__main__:Batch 19/5000 complete
DEBUG:__main__:Batch 20/5000 complete
DEBUG:__main__:Batch 21/5000 complete
DEBUG:__main__:Batch 22/5000 complete
DEBUG:__main__:Batch 23/5000 complete
DEBUG:__main__:Batch 24/5000 complete
DEBUG:__main__:Batch 25/5000 complete
DEBUG:__main__:Batch 26/5000 complete

In [36]:
import os

with open("openwakeword/openwakeword/train.py", "r") as f:
    lines = f.readlines()

# Lihat sekitar baris 644-655 untuk konteks
for i, line in enumerate(lines[640:660], start=641):
    print(f"{i}: {repr(line)}")

641: '    args = parser.parse_args()\n'
642: "    config = yaml.load(open(args.training_config, 'r').read(), yaml.Loader)\n"
643: '\n'
644: '    # imports Piper for synthetic sample generation\n'
645: '    sys.path.insert(0, os.path.abspath(config["piper_sample_generator_path"]))\n'
646: '    from piper_sample_generator.__main__ import generate_samples  # patched\n'
647: '\n'
648: '    # Define output locations\n'
649: '    config["output_dir"] = os.path.abspath(config["output_dir"])\n'
650: '    if not os.path.exists(config["output_dir"]):\n'
651: '        os.mkdir(config["output_dir"])\n'
652: '    if not os.path.exists(os.path.join(config["output_dir"], config["model_name"])):\n'
653: '        os.mkdir(os.path.join(config["output_dir"], config["model_name"]))\n'
654: '\n'
655: '    positive_train_output_dir = os.path.join(config["output_dir"], config["model_name"], "positive_train")\n'
656: '    positive_test_output_dir = os.path.join(config["output_dir"], config["model_name"], "pos

In [37]:
# Tulis ulang train.py dengan patch yang benar di baris yang tepat
with open("openwakeword/openwakeword/train.py", "r") as f:
    content = f.read()

# Reset dulu kalau patch sebelumnya rusak
import re

# Hapus patch lama yang mungkin sudah masuk
content = re.sub(
    r'import sys as _sys\n_sys\.path\.insert\(0, "\./piper-sample-generator"\)\n',
    '',
    content
)

# Ganti import yang salah dengan yang benar (single line replacement)
old = "from generate_samples import generate_samples"
new = "from piper_sample_generator.__main__ import generate_samples  # patched"

if old in content:
    content = content.replace(old, new)
    print("✅ Import di-patch")
elif "patched" in content:
    print("✅ Sudah di-patch sebelumnya")
else:
    print("⚠️  String tidak ditemukan, isi baris sekitar 646:")
    for i, l in enumerate(content.splitlines()[640:660], start=641):
        print(f"  {i}: {repr(l)}")

with open("openwakeword/openwakeword/train.py", "w") as f:
    f.write(content)

# Tambahkan piper-sample-generator ke sys.path via .pth file
# supaya import bisa resolve tanpa ubah train.py lebih jauh
import site
pth_path = os.path.join(site.getsitepackages()[0], "piper_sampler.pth")
with open(pth_path, "w") as f:
    f.write("/content/piper-sample-generator\n")
print(f"✅ sys.path entry added: {pth_path}")

✅ Sudah di-patch sebelumnya
✅ sys.path entry added: /usr/local/lib/python3.12/dist-packages/piper_sampler.pth


In [38]:
# Verifikasi patch bersih
with open("openwakeword/openwakeword/train.py", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[640:660], start=641):
    print(f"{i}: {repr(line)}")

641: '    args = parser.parse_args()\n'
642: "    config = yaml.load(open(args.training_config, 'r').read(), yaml.Loader)\n"
643: '\n'
644: '    # imports Piper for synthetic sample generation\n'
645: '    sys.path.insert(0, os.path.abspath(config["piper_sample_generator_path"]))\n'
646: '    from piper_sample_generator.__main__ import generate_samples  # patched\n'
647: '\n'
648: '    # Define output locations\n'
649: '    config["output_dir"] = os.path.abspath(config["output_dir"])\n'
650: '    if not os.path.exists(config["output_dir"]):\n'
651: '        os.mkdir(config["output_dir"])\n'
652: '    if not os.path.exists(os.path.join(config["output_dir"], config["model_name"])):\n'
653: '        os.mkdir(os.path.join(config["output_dir"], config["model_name"]))\n'
654: '\n'
655: '    positive_train_output_dir = os.path.join(config["output_dir"], config["model_name"], "positive_train")\n'
656: '    positive_test_output_dir = os.path.join(config["output_dir"], config["model_name"], "pos

In [39]:
import os
import sys
import shlex

# --- 1. INSTALL MISSING EXPORT DEPENDENCY ---
print('--- Installing onnxscript for model export ---')
!pip install -q onnxscript

# --- 2. PREPARE WORKSPACE ---
print('--- Preparing workspace ---')
model_name = 'hey_pisces'
current_dir = os.path.abspath(os.getcwd())
model_dir = os.path.join(current_dir, 'my_custom_model', model_name)

# --- 3. EXECUTE TRAINING ---
config_path = os.path.join(current_dir, 'my_custom_model', f"{model_name}_config.yaml")
openwakeword_root = os.path.join(current_dir, 'openwakeword')
full_pp = ":".join([current_dir, os.path.join(current_dir, 'piper-sample-generator'), openwakeword_root])
train_script = os.path.join(openwakeword_root, 'openwakeword/train.py')

print(f'--- Starting Final Training & Export for {model_name} ---')
!PYTHONPATH={full_pp} python3 {shlex.quote(train_script)} --training_config {shlex.quote(config_path)} --train_model

--- Installing onnxscript for model export ---
--- Preparing workspace ---
--- Starting Final Training & Export for hey_pisces ---
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 751, in <module>
    sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 798, in numpy.random.mtrand.RandomState.randint
  File "numpy/random/_bounded_integers.pyx", line 1334, in numpy.random._bounded_integers._rand_int64
ValueError: high <= 0


In [40]:
# --- NEW CELL: TFLITE CONVERSION ---
import os
import shutil

# 1. Install modern conversion tools and missing dependencies
print('--- Preparing conversion environment ---')
# onnx2tf requires onnx_graphsurgeon
!pip install -q onnx2tf siso onnx==1.16.1
!pip install -q onnx-graphsurgeon --index-url https://pypi.ngc.nvidia.com

# 2. Define Paths
model_name = 'hey_pisces'
output_dir = os.path.join(os.getcwd(), 'my_custom_model')
onnx_path = os.path.join(output_dir, f'{model_name}.onnx')
tflite_path = os.path.join(output_dir, f'{model_name}.tflite')

# 3. Perform Conversion
if os.path.exists(onnx_path):
    print(f'--- Converting {onnx_path} to TFLite ---')
    # Use onnx2tf CLI for best compatibility with Python 3.12
    !onnx2tf -i {onnx_path} -o {output_dir} --non_verbose

    # onnx2tf generates filenames like model_float32.tflite
    generated = os.path.join(output_dir, f'{model_name}_float32.tflite')
    if os.path.exists(generated):
        shutil.move(generated, tflite_path)
        print(f'✨ SUCCESS: Model exported to {tflite_path}')
    elif os.path.exists(tflite_path):
        print(f'✨ SUCCESS: Model exists at {tflite_path}')
    else:
        print('❌ Conversion finished but output file was not found.')
else:
    print(f'❌ ONNX model not found at {onnx_path}.')

# 4. Final verification of both files
print("\n--- Final File Check ---")
for ext in ['onnx', 'tflite']:
    p = os.path.join(output_dir, f'{model_name}.{ext}')
    if os.path.exists(p):
        print(f'✅ Ready for download: {p} ({os.path.getsize(p)/1024:.1f} KB)')
    else:
        print(f'❌ Missing: {p}')

--- Preparing conversion environment ---
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnxscript 0.7.1 requires onnx>=1.17, but you have onnx 1.16.1 which is incompatible.
❌ ONNX model not found at /content/my_custom_model/hey_pisces.onnx.

--- Final File Check ---
❌ Missing: /content/my_custom_model/hey_pisces.onnx
❌ Missing: /content/my_custom_model/hey_pisces.tflite


In [41]:
import os
from pathlib import Path

model_name = 'hey_pisces'
model_dir = f'/content/my_custom_model/{model_name}'
output_dir = '/content/my_custom_model'

print('--- Final Model and Feature Verification ---')

# Check for feature files
features = ['positive_features_train.npy', 'positive_features_test.npy', 'negative_features_train.npy', 'negative_features_test.npy']
for f in features:
    path = os.path.join(model_dir, f)
    status = '✅ Found' if os.path.exists(path) else '❌ Missing'
    size = f'({os.path.getsize(path)/1024**2:.2f} MB)' if os.path.exists(path) else ''
    print(f'{status}: {f} {size}')

# Check for final exported models
models = [f'{model_name}.onnx', f'{model_name}.tflite']
for m in models:
    path = os.path.join(output_dir, m)
    status = '✨ SUCCESS' if os.path.exists(path) else '⏳ Not generated yet'
    print(f'{status}: {m}')

--- Final Model and Feature Verification ---
❌ Missing: positive_features_train.npy 
❌ Missing: positive_features_test.npy 
❌ Missing: negative_features_train.npy 
❌ Missing: negative_features_test.npy 
⏳ Not generated yet: hey_pisces.onnx
⏳ Not generated yet: hey_pisces.tflite


In [42]:
import os
from pathlib import Path

model_name = 'hey_pisces'
output_dir = os.path.abspath('my_custom_model')
pos_train_dir = os.path.join(output_dir, model_name, 'positive_train')

# Check positive clips
if os.path.exists(pos_train_dir):
    clips = list(Path(pos_train_dir).glob('*.wav'))
    print(f'✅ Found {len(clips)} positive training clips in {pos_train_dir}')
else:
    print(f'❌ Positive training directory not found: {pos_train_dir}')

# Check for generated models
onnx_path = os.path.join(output_dir, f'{model_name}.onnx')
tflite_path = os.path.join(output_dir, f'{model_name}.tflite')

if os.path.exists(onnx_path):
    print(f'✅ Model exported to ONNX: {onnx_path}')
else:
    print(f'❌ ONNX model not found yet.')

if os.path.exists(tflite_path):
    print(f'✅ Model exported to TFLite: {tflite_path}')
else:
    print(f'❌ TFLite model not found yet.')

❌ Positive training directory not found: /content/my_custom_model/hey_pisces/positive_train
❌ ONNX model not found yet.
❌ TFLite model not found yet.


In [43]:
# Re-verify the content of train.py around line 748, using repr() for exact string matching
with open("openwakeword/openwakeword/train.py", "r") as f:
    train_py_content_recheck = f.readlines()

print("Content of train.py around original line 748 (using repr()):")
# The original line 748 is now likely at index 747 in the list.
# Let's check a few lines around it.
for i, line in enumerate(train_py_content_recheck[745:750], start=746):
    print(f"{i}: {repr(line)}")


Content of train.py around original line 748 (using repr()):
746: "    # and setting to 32000 when the median + 750 ms is close to that, as it's a good default value\n"
747: '    n = 50  # sample size\n'
748: '    positive_clips = [str(i) for i in Path(positive_test_output_dir).glob("*.wav")]\n'
749: '    duration_in_samples = []\n'
750: '    for i in range(n):\n'


In [44]:
# Verify the content of train.py after patching
with open("openwakeword/openwakeword/train.py", "r") as f:
    train_py_content_after_patch = f.readlines()

print("Content of train.py around patched line (748) after the patching cell:")
# Adjust range based on previous inserts. Original line 748 would now be 750 (748 + 2 inserts)
for i, line in enumerate(train_py_content_after_patch[740:760], start=741):
    print(f"{i}: {line.strip()}")


Content of train.py around patched line (748) after the patching cell:
741: torch.cuda.empty_cache()
742: else:
743: logging.warning(f"Skipping generation of negative clips for testing, as ~{config['n_samples_val']} already exist")
744: 
745: # Set the total length of the training clips based on the ~median generated clip duration, rounding to the nearest 1000 samples
746: # and setting to 32000 when the median + 750 ms is close to that, as it's a good default value
747: n = 50  # sample size
748: positive_clips = [str(i) for i in Path(positive_test_output_dir).glob("*.wav")]
749: duration_in_samples = []
750: for i in range(n):
751: sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
752: duration_in_samples.append(len(dat))
753: 
754: config["total_length"] = int(round(np.median(duration_in_samples)/1000)*1000) + 12000  # add 750 ms to clip duration as buffer
755: if config["total_length"] < 32000:
756: config["total_length"] = 32000  # set a mi

In [45]:
# Cek struktur piper-sample-generator
!find piper-sample-generator -name "*.py" | grep -i "generate\|sample" | head -20
!echo "---"
!grep -n "def generate_samples" piper-sample-generator/piper_sample_generator/*.py 2>/dev/null || \
 grep -rn "def generate_samples" piper-sample-generator/ 2>/dev/null | head -5


piper-sample-generator/piper_train/vits/config.py
piper-sample-generator/piper_train/vits/transforms.py
piper-sample-generator/piper_train/vits/attentions.py
piper-sample-generator/piper_train/vits/dataset.py
piper-sample-generator/piper_train/vits/commons.py
piper-sample-generator/piper_train/vits/mel_processing.py
piper-sample-generator/piper_train/vits/modules.py
piper-sample-generator/piper_train/vits/utils.py
piper-sample-generator/piper_train/vits/__init__.py
piper-sample-generator/piper_train/vits/losses.py
piper-sample-generator/piper_train/vits/models.py
piper-sample-generator/piper_train/vits/lightning.py
piper-sample-generator/piper_train/vits/wavfile.py
piper-sample-generator/piper_train/norm_audio/trim.py
piper-sample-generator/piper_train/norm_audio/vad.py
piper-sample-generator/piper_train/norm_audio/__init__.py
piper-sample-generator/piper_train/__main__.py
piper-sample-generator/piper_train/_resources.py
piper-sample-generator/piper_train/__init__.py
piper-sample-gener

In [46]:
import os

checks = {
    "piper-sample-generator clone":     os.path.exists("piper-sample-generator"),
    "model .pt downloaded":             os.path.exists("piper-sample-generator/models/en_US-libritts_r-medium.pt"),
    "openwakeword clone":               os.path.exists("openwakeword"),
    "embedding_model.onnx":             os.path.exists("openwakeword/openwakeword/resources/models/embedding_model.onnx"),
    "melspectrogram.onnx":              os.path.exists("openwakeword/openwakeword/resources/models/melspectrogram.onnx"),
    "positive clips dir":               os.path.exists("my_custom_model/hey_pisces/positive_clips_train"), # Changed from hey_sebastian and added _train
    "positive clips count":             len(os.listdir("my_custom_model/hey_pisces/positive_clips_train")) if os.path.exists("my_custom_model/hey_pisces/positive_clips_train") else 0,
    "positive_features_train.npy":      os.path.exists("my_custom_model/hey_pisces/positive_features_train.npy"),
    "positive_features_test.npy":       os.path.exists("my_custom_model/hey_pisces/positive_features_test.npy"),
    "hey_pisces.onnx":                  os.path.exists("my_custom_model/hey_pisces.onnx"),
}

for k, v in checks.items():
    print(f"{'✅' if v else '❌'} {k}: {v}")

✅ piper-sample-generator clone: True
✅ model .pt downloaded: True
✅ openwakeword clone: True
✅ embedding_model.onnx: True
✅ melspectrogram.onnx: True
✅ positive clips dir: True
✅ positive clips count: 5000
❌ positive_features_train.npy: False
❌ positive_features_test.npy: False
❌ hey_pisces.onnx: False


In [47]:
positive_clips_dir = "my_custom_model/hey_pisces/positive_clips_train"
!ls -l "{positive_clips_dir}"

Output streaming akan dipotong hingga 5000 baris terakhir.
-rw-r--r-- 1 root root 42540 Jun 30 15:05 0.wav
-rw-r--r-- 1 root root 51756 Jun 30 15:08 1000.wav
-rw-r--r-- 1 root root 43564 Jun 30 15:08 1001.wav
-rw-r--r-- 1 root root 56364 Jun 30 15:08 1002.wav
-rw-r--r-- 1 root root 56364 Jun 30 15:08 1003.wav
-rw-r--r-- 1 root root 52268 Jun 30 15:09 1004.wav
-rw-r--r-- 1 root root 54828 Jun 30 15:09 1005.wav
-rw-r--r-- 1 root root 52780 Jun 30 15:09 1006.wav
-rw-r--r-- 1 root root 50220 Jun 30 15:09 1007.wav
-rw-r--r-- 1 root root 36908 Jun 30 15:09 1008.wav
-rw-r--r-- 1 root root 44588 Jun 30 15:09 1009.wav
-rw-r--r-- 1 root root 35372 Jun 30 15:06 100.wav
-rw-r--r-- 1 root root 39980 Jun 30 15:09 1010.wav
-rw-r--r-- 1 root root 41516 Jun 30 15:09 1011.wav
-rw-r--r-- 1 root root 45612 Jun 30 15:09 1012.wav
-rw-r--r-- 1 root root 39468 Jun 30 15:09 1013.wav
-rw-r--r-- 1 root root 32812 Jun 30 15:09 1014.wav
-rw-r--r-- 1 root root 34860 Jun 30 15:09 1015.wav
-rw-r--r-- 1 root root 3588

In [48]:
import os

# Membaca kembali file train.py untuk memeriksa bagian yang relevan
with open("openwakeword/openwakeword/train.py", "r") as f:
    train_py_content = f.readlines()

# Mencari di sekitar baris 751 dan bagian di mana positive_clips diinisialisasi
print("Code in train.py around line 751:")
for i, line in enumerate(train_py_content[740:760], start=741):
    print(f"{i}: {line.strip()}")

print("\nSearching for 'positive_clips' initialization:")
positive_clips_init_lines = []
for i, line in enumerate(train_py_content):
    if "positive_clips = " in line or "positive_clips.append" in line:
        # Menampilkan beberapa baris di sekitar inisialisasi untuk konteks
        start_line = max(0, i - 5)
        end_line = min(len(train_py_content), i + 5)
        print(f"Found 'positive_clips' reference around line {i+1}:")
        for j in range(start_line, end_line):
            print(f"{j+1}: {train_py_content[j].strip()}")
        break # Asumsi hanya ada satu inisialisasi utama untuk positive_clips yang menyebabkan masalah


Code in train.py around line 751:
741: torch.cuda.empty_cache()
742: else:
743: logging.warning(f"Skipping generation of negative clips for testing, as ~{config['n_samples_val']} already exist")
744: 
745: # Set the total length of the training clips based on the ~median generated clip duration, rounding to the nearest 1000 samples
746: # and setting to 32000 when the median + 750 ms is close to that, as it's a good default value
747: n = 50  # sample size
748: positive_clips = [str(i) for i in Path(positive_test_output_dir).glob("*.wav")]
749: duration_in_samples = []
750: for i in range(n):
751: sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
752: duration_in_samples.append(len(dat))
753: 
754: config["total_length"] = int(round(np.median(duration_in_samples)/1000)*1000) + 12000  # add 750 ms to clip duration as buffer
755: if config["total_length"] < 32000:
756: config["total_length"] = 32000  # set a minimum of 32000 samples (2 seconds)
75